**Objetivo:** Validar as matrizes de conectividade funcional quanto à dimensão, presença de valores inválidos, simetria, intervalo dos coeficientes de correlação e correspondência dos participantes com os dados fenotípicos.

**Entradas:** `Data/conectivity/connectivity_matrices_abide.pkl` (Caderno 04) e `Data/raw/phenotypic_abide.csv` (Caderno 02).

**Processamento:** Verificação individual das matrizes de conectividade. Uma matriz somente é considerada válida quando apresenta dimensão 200 × 200, valores finitos, simetria e coeficientes no intervalo esperado da correlação de Pearson. Também é verificada a correspondência dos identificadores dos participantes com os dados fenotípicos.

**Saídas:** `Data/conectivity/connectivity_matrices_abide_validado.pkl` e, quando houver problemas, `Docs/05_relatorio_exclusoes.csv`.

In [6]:
# ------------------------------------------------------------
# Configuração do projeto (execução local — VS Code)
# Garante que a raiz do projeto esteja no sys.path, funcionando tanto se o
# diretório de trabalho do kernel for notebooks/ quanto a raiz do projeto.
# ------------------------------------------------------------
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

Projeto (BASE_DIR) = /home/maria/TCC_TEA


In [7]:
# ============================================================
# CARREGAMENTO DAS MATRIZES E DOS DADOS FENOTÍPICOS
# ============================================================

exigir_arquivo(
    PATHS['conectivity']
    / 'connectivity_matrices_abide.pkl',
    '04_Conectividade_ABIDE.ipynb'
)

exigir_arquivo(
    PATHS['raw']
    / 'phenotypic_abide.csv',
    '02_Download_ABIDE.ipynb'
)

import pickle
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Carregar matrizes de conectividade
# ------------------------------------------------------------

conn_path = (
    PATHS['conectivity']
    / 'connectivity_matrices_abide.pkl'
)

with open(
    conn_path,
    'rb'
) as f:

    conn_dict = pickle.load(f)

# ------------------------------------------------------------
# Padronizar SUB_ID das matrizes
# ------------------------------------------------------------

conn_dict = {
    str(sub_id): np.asarray(mat)
    for sub_id, mat in conn_dict.items()
}

# ------------------------------------------------------------
# Carregar dados fenotípicos
# ------------------------------------------------------------

phenotypic = pd.read_csv(
    PATHS['raw']
    / 'phenotypic_abide.csv'
)

phenotypic['SUB_ID'] = (
    phenotypic['SUB_ID']
    .astype(str)
)

phenotypic = (
    phenotypic
    .set_index('SUB_ID')
)

# ------------------------------------------------------------
# Conferência
# ------------------------------------------------------------

print(
    f'{len(conn_dict)} sujeitos carregados '
    'para validação.'
)

print(
    f'{len(phenotypic)} registros fenotípicos carregados.'
)

# ------------------------------------------------------------
# Correspondência dos IDs
# ------------------------------------------------------------

ids_matrizes = set(
    conn_dict.keys()
)

ids_phenotypic = set(
    phenotypic.index
)

sem_fenotipo = (
    ids_matrizes -
    ids_phenotypic
)

sem_matriz = (
    ids_phenotypic -
    ids_matrizes
)

print()
print(
    'Matrizes sem registro fenotípico:',
    len(sem_fenotipo)
)

print(
    'Participantes sem matriz:',
    len(sem_matriz)
)

871 sujeitos carregados para validação.
871 registros fenotípicos carregados.

Matrizes sem registro fenotípico: 0
Participantes sem matriz: 0


#### Validação da Matriz

In [8]:
# ============================================================
# VALIDAÇÃO DAS MATRIZES DE CONECTIVIDADE
# ============================================================

def validar_matriz(
    mat,
    esperado_shape=(200, 200),
    tol=1e-6
):

    """
    Retorna uma lista de problemas encontrados na matriz.
    Uma lista vazia significa que a matriz passou na validação.
    """

    problemas = []

    mat = np.asarray(mat)

    # --------------------------------------------------------
    # Dimensão
    # --------------------------------------------------------

    if mat.shape != esperado_shape:

        problemas.append(
            f'shape inesperado: {mat.shape}'
        )

        # Não continuar com testes que dependem
        # de uma matriz 200 × 200.

        return problemas

    # --------------------------------------------------------
    # Valores finitos
    # --------------------------------------------------------

    if not np.isfinite(mat).all():

        problemas.append(
            'contém NaN ou Inf'
        )

    # --------------------------------------------------------
    # Simetria
    # --------------------------------------------------------

    if not np.allclose(
        mat,
        mat.T,
        atol=tol
    ):

        problemas.append(
            'matriz não é simétrica'
        )

    # --------------------------------------------------------
    # Intervalo da correlação de Pearson
    # --------------------------------------------------------

    if (
        mat.min() < -1 - tol
        or
        mat.max() > 1 + tol
    ):

        problemas.append(
            'valores fora de [-1, 1]: '
            f'min={mat.min():.6f}, '
            f'max={mat.max():.6f}'
        )

    return problemas


# ============================================================
# APLICAÇÃO DA VALIDAÇÃO
# ============================================================

relatorio = []

sujeitos_validos = {}

for sub_id, mat in conn_dict.items():

    problemas = validar_matriz(
        mat
    )

    if sub_id in phenotypic.index:

        site = phenotypic.loc[
            sub_id,
            'SITE_ID'
        ]

    else:

        site = 'desconhecido'

        problemas.append(
            'SUB_ID não encontrado nos dados fenotípicos'
        )

    if problemas:

        relatorio.append(
            {
                'SUB_ID': sub_id,
                'SITE_ID': site,
                'problemas': '; '.join(
                    problemas
                )
            }
        )

    else:

        sujeitos_validos[sub_id] = (
            np.asarray(mat)
        )

# ============================================================
# RESUMO
# ============================================================

print(
    f'Sujeitos válidos: '
    f'{len(sujeitos_validos)} / '
    f'{len(conn_dict)}'
)

print(
    f'Sujeitos excluídos: '
    f'{len(relatorio)}'
)

Sujeitos válidos: 871 / 871
Sujeitos excluídos: 0


In [9]:
# ============================================================
# RELATÓRIO DA VALIDAÇÃO
# ============================================================

df_relatorio = pd.DataFrame(
    relatorio
)

print(
    '=========================================='
)

print(
    'RESULTADO DA VALIDAÇÃO'
)

print(
    '=========================================='
)

print(
    'Total de matrizes:',
    len(conn_dict)
)

print(
    'Matrizes válidas:',
    len(sujeitos_validos)
)

print(
    'Matrizes excluídas:',
    len(relatorio)
)

# ------------------------------------------------------------
# Caso existam exclusões
# ------------------------------------------------------------

if not df_relatorio.empty:

    print()
    print(
        'Exclusões por site:'
    )

    print(
        df_relatorio[
            'SITE_ID'
        ].value_counts()
    )

    print()
    print(
        'Problemas encontrados:'
    )

    print(
        df_relatorio[
            'problemas'
        ].value_counts()
    )

    relatorio_path = (
        PATHS['docs']
        / '05_relatorio_exclusoes.csv'
    )

    df_relatorio.to_csv(
        relatorio_path,
        index=False,
        encoding='utf-8'
    )

    print()
    print(
        'Relatório salvo em:',
        relatorio_path
    )

else:

    print()
    print(
        'Nenhum sujeito excluído — '
        'todas as matrizes passaram na validação.'
    )

RESULTADO DA VALIDAÇÃO
Total de matrizes: 871
Matrizes válidas: 871
Matrizes excluídas: 0

Nenhum sujeito excluído — todas as matrizes passaram na validação.


In [10]:
# ============================================================
# SALVAR MATRIZES VALIDADAS
# ============================================================

out_path = (
    PATHS['conectivity']
    / 'connectivity_matrices_abide_validado.pkl'
)

with open(
    out_path,
    'wb'
) as f:

    pickle.dump(
        sujeitos_validos,
        f
    )

print(
    'Matrizes validadas salvas em:',
    out_path
)

print(
    'Quantidade de participantes:',
    len(sujeitos_validos)
)

# ------------------------------------------------------------
# Verificação final
# ------------------------------------------------------------

if sujeitos_validos:

    primeira_matriz = (
        next(
            iter(
                sujeitos_validos.values()
            )
        )
    )

    print(
        'Shape das matrizes:',
        primeira_matriz.shape
    )

Matrizes validadas salvas em: /home/maria/TCC_TEA/Data/conectivity/connectivity_matrices_abide_validado.pkl
Quantidade de participantes: 871
Shape das matrizes: (200, 200)
